In [2]:
import warnings
warnings.filterwarnings('ignore')

In [5]:
import keras
from keras import models, layers
from keras.layers import Dense, Conv2D, Flatten
from keras import Sequential
from keras.datasets import mnist

In [6]:
(x_train, y_train), (x_test, y_test) = mnist.load_data()

In [9]:
model = models.Sequential([
    # in CNN we use Conv2D, just like we were doing Dense in ANN
    layers.Conv2D(32, kernel_size=(3, 3), padding='same', activation='relu', input_shape=(28, 28, 1)),
    layers.MaxPooling2D(pool_size=(2, 2)), # Dimensions down to (14, 14, 32)

    layers.Conv2D(64, kernel_size=(3, 3), padding='same', activation='relu'),
    layers.MaxPooling2D(pool_size=(2, 2)), # Dimensions down to (7, 7, 64)

    layers.Conv2D(64, kernel_size=(3, 3), padding='same', activation='relu'),
    layers.MaxPooling2D(pool_size=(2, 2)), 
    
    # Flattening the downsampled maps (7 * 7 * 64 = 3,136 elements)
    layers.Flatten(),
    
    # Fully Connected Dense Layers
    layers.Dense(128, activation='relu'),    # Fixed hidden layer activation to ReLU
    layers.Dense(10, activation='softmax')   # Output layer (assumes 10 target classes, e.g., MNIST)
    
])

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.


**Max Pooling** is a downsampling operation commonly used in Convolutional Neural Networks (CNNs) to reduce the spatial size of image representations. It helps the network focus on the most dominant features while cutting down on computational costs.

Max Pooling uses a small grid window (usually 2x2 pixels) that slides across a feature map with a set step size (stride).

At each position, the operation looks at all the numbers inside the window, selects the maximum value, and discards the rest.

## The Pros (Advantages)

1. **Significant Dimensionality Reduction** : Max Pooling aggressively reduces the spatial size (width and height) of feature maps. For instance, a standard $2 \times 2$ window with a stride of 2 discards 75% of the activations, which directly scales down the computational load, memory footprint, and training time for subsequent layers.

2. **Translation Invariance** : It makes the network highly robust to minor spatial shifts, rotations, or distortions of features within an image. If a specific feature (like a sharp corner) moves by a pixel or two, the maximum value within that local window often stays identical, ensuring the network registers the feature regardless of its exact location.

3. **Feature Sharpness and Noise Reduction** : By selecting only the maximum activation value, the operation acts as a prominent feature selector. It extracts the most dominant structural signals (such as stark contrast edges) while naturally discarding lower, sub-optimal background noise or minor variations in illumination.

4. **Mitigation of Overfitting** : By systematically reducing the overall parameter count right before flattening data into dense layers, it acts as a form of regularization. Fewer parameters mean the network is less likely to memorize pixel-perfect details of the training dataset, enhancing its ability to generalize to new data.


## The Cons (Disadvantages)

1. **Severe Loss of Spatial Information** : Max Pooling is a destructive operation. Because it selects only a single maximum value from a grid window, it discards the other 75% of the data points. This means the exact pixel-level coordinates and spatial relationships of internal components are permanently lost.

2. **Ineffectiveness for Dense Prediction Tasks** : For applications that require pixel-perfect precision—such as **Semantic Segmentation** (e.g., identifying the exact boundary of a medical tumor or a pedestrian in self-driving cars) or **Image Reconstruction**—the heavy downsampling caused by Max Pooling destroys the fine structural alignments needed to produce accurate outputs.

In [10]:
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_2 (Conv2D)               │ (None, 28, 28, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 14, 14, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 14, 14, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 7, 7, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 7, 7, 64)       │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 3, 3, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 576)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 130,890 (511.29 KB)

 Trainable params: 130,890 (511.29 KB)

 Non-trainable params: 0 (0.00 B)

As you can see above there is information loss due to `max_pooling`

In [11]:
model = models.Sequential([
    # in CNN we use Conv2D, just like we were doing Dense in ANN
    layers.Conv2D(32, kernel_size=(3, 3), padding='same', activation='relu', input_shape=(28, 28, 1)),
    layers.Conv2D(64, kernel_size=(3, 3), padding='same', activation='relu'),
    layers.Conv2D(64, kernel_size=(3, 3), padding='same', activation='relu'),
    
    # Flattening the downsampled maps (7 * 7 * 64 = 3,136 elements)
    layers.Flatten(),
    
    # Fully Connected Dense Layers
    layers.Dense(128, activation='relu'),    # Fixed hidden layer activation to ReLU
    layers.Dense(10, activation='softmax')   # Output layer (assumes 10 target classes, e.g., MNIST)
    
])

In [12]:
model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_5 (Conv2D)               │ (None, 28, 28, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_6 (Conv2D)               │ (None, 28, 28, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_7 (Conv2D)               │ (None, 28, 28, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_2 (Flatten)             │ (None, 50176)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 128)            │     6,422,656 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 6,479,690 (24.72 MB)

 Trainable params: 6,479,690 (24.72 MB)

 Non-trainable params: 0 (0.00 B)

As you can see above there is no information loos